# Entity Identification Pipeline

Compare **pre-trained (retrieval)** vs **fine-tuned (with description augmentation)**.

**Data:**
- `user_queries.csv`: Queries with JSON containing field references
- `fields_description.csv`: Field descriptions (matched by field_name)

**Key idea:** Extract field descriptions from JSON and use them for training/retrieval.

In [ ]:
import pandas as pd
import numpy as np
import json
import ast
from typing import List, Dict, Tuple, Set
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, hamming_loss
)

import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModel, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASE_MODEL = "distilbert-base-uncased"
print(f"Device: {DEVICE}, Model: {BASE_MODEL}")

## 1. Load and Process Data

In [ ]:
# Load data
user_queries_df = pd.read_csv('user_queries.csv')
fields_df = pd.read_csv('fields_description.csv')

# Build field_name → description lookup
FIELD_TO_DESC = dict(zip(fields_df['field_name'], fields_df['description']))

print(f"User queries: {len(user_queries_df)}")
print(f"Field descriptions: {len(FIELD_TO_DESC)}")

In [ ]:
def parse_json(json_str: str) -> dict:
    """Parse JSON string."""
    try:
        return json.loads(json_str)
    except:
        try:
            return ast.literal_eval(json_str)
        except:
            return {}

def extract_field_names(json_obj: dict) -> Set[str]:
    """Extract all 'name' fields from JSON statements."""
    names = set()
    
    def search(obj):
        if isinstance(obj, dict):
            if 'name' in obj:
                names.add(obj['name'])
            for v in obj.values():
                search(v)
        elif isinstance(obj, list):
            for item in obj:
                search(item)
    
    search(json_obj)
    return names

def extract_entities(json_obj: dict) -> List[str]:
    """Extract entities from entityType and relationTargetType."""
    entities = set()
    if 'entityType' in json_obj:
        entities.add(json_obj['entityType'])
    
    def search(obj):
        if isinstance(obj, dict):
            if 'relationTargetType' in obj:
                targets = obj['relationTargetType']
                entities.update(targets if isinstance(targets, list) else [targets])
            for v in obj.values():
                search(v)
        elif isinstance(obj, list):
            for item in obj:
                search(item)
    
    search(json_obj)
    return sorted(list(entities))

def get_descriptions_for_query(json_obj: dict) -> str:
    """Get combined descriptions for all field names in a query."""
    field_names = extract_field_names(json_obj)
    descriptions = []
    for name in field_names:
        if name in FIELD_TO_DESC:
            descriptions.append(FIELD_TO_DESC[name])
    return " ".join(descriptions)

In [ ]:
# Process user queries
user_queries_df['parsed_json'] = user_queries_df['json'].apply(parse_json)
user_queries_df['entities'] = user_queries_df['parsed_json'].apply(extract_entities)
user_queries_df['descriptions'] = user_queries_df['parsed_json'].apply(get_descriptions_for_query)

# Show example
print("Example:")
print(f"Query: {user_queries_df['question'].iloc[0]}")
print(f"Entities: {user_queries_df['entities'].iloc[0]}")
print(f"Descriptions: {user_queries_df['descriptions'].iloc[0][:200]}...")

In [ ]:
# Entity distribution
all_entities = [e for ents in user_queries_df['entities'] for e in ents]
ALL_ENTITIES = sorted(set(all_entities))
print(f"Entities ({len(ALL_ENTITIES)}): {ALL_ENTITIES}")
print("\nDistribution:")
for e, c in Counter(all_entities).most_common():
    print(f"  {e}: {c}")

## 2. Train/Test Split

In [ ]:
# Encode labels
mlb = MultiLabelBinarizer(classes=ALL_ENTITIES)
y_encoded = mlb.fit_transform(user_queries_df['entities'])

# Split indices (80/20)
indices = list(range(len(user_queries_df)))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)

# Training data
train_queries = user_queries_df['question'].iloc[train_idx].tolist()
train_descriptions = user_queries_df['descriptions'].iloc[train_idx].tolist()
train_entities = [user_queries_df['entities'].iloc[i] for i in train_idx]
y_train = y_encoded[train_idx]

# Test data (queries only - simulates inference)
test_queries = user_queries_df['question'].iloc[test_idx].tolist()
y_test = y_encoded[test_idx]

print(f"Train: {len(train_queries)}, Test: {len(test_queries)}")

## 3. Shared Components

In [ ]:
class Embedder:
    """Compute embeddings using pre-trained model."""
    
    def __init__(self, model_name: str = BASE_MODEL):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.to(DEVICE)
        self.model.eval()
        for p in self.model.parameters():
            p.requires_grad = False
    
    def embed(self, text: str) -> np.ndarray:
        """Get mean-pooled embedding."""
        inputs = self.tokenizer(text, return_tensors='pt', truncation=True,
                                padding=True, max_length=256)
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        with torch.no_grad():
            out = self.model(**inputs)
            mask = inputs['attention_mask'].unsqueeze(-1)
            pooled = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        return pooled.cpu().numpy()[0]
    
    def embed_batch(self, texts: List[str], show_progress: bool = True) -> np.ndarray:
        """Embed multiple texts."""
        embeddings = []
        for i, text in enumerate(texts):
            if show_progress and i % 50 == 0:
                print(f"  Embedding {i+1}/{len(texts)}...")
            embeddings.append(self.embed(text))
        return np.array(embeddings)


def evaluate(y_true: np.ndarray, y_pred: np.ndarray, labels: List[str], name: str) -> Dict:
    """Compute and display metrics."""
    metrics = {
        'exact_match': accuracy_score(y_true, y_pred),
        'f1_micro': f1_score(y_true, y_pred, average='micro', zero_division=0),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'precision': precision_score(y_true, y_pred, average='micro', zero_division=0),
        'recall': recall_score(y_true, y_pred, average='micro', zero_division=0),
        'hamming_loss': hamming_loss(y_true, y_pred),
    }
    print(f"\n{'='*50}\n{name}\n{'='*50}")
    for k, v in metrics.items():
        print(f"{k:15}: {v:.4f}")
    print(classification_report(y_true, y_pred, target_names=labels, zero_division=0))
    return metrics

## 4. Approach 1: Pre-trained (Retrieval)

Build database of (query + descriptions) embeddings. At inference, find most similar.

In [ ]:
class RetrievalClassifier:
    """Pre-trained retrieval-based classifier."""
    
    def __init__(self):
        self.embedder = Embedder(BASE_MODEL)
        self.db_embeddings = None
        self.db_entities = None
    
    def build_database(self, queries: List[str], descriptions: List[str], 
                       entities: List[List[str]]):
        """Build retrieval database from training data."""
        print("Building retrieval database...")
        # Combine query + descriptions for each entry
        combined = [f"{q} {d}" for q, d in zip(queries, descriptions)]
        self.db_embeddings = self.embedder.embed_batch(combined)
        self.db_entities = entities
        print(f"Database built: {len(self.db_embeddings)} entries")
    
    def predict(self, queries: List[str], k: int = 1) -> List[List[str]]:
        """Predict by finding most similar entries in database using k-NN voting."""
        print(f"Predicting {len(queries)} queries with k={k}...")
        predictions = []

        # Normalize database embeddings
        db_norm = self.db_embeddings / (np.linalg.norm(self.db_embeddings, axis=1, keepdims=True) + 1e-9)

        for i, query in enumerate(queries):
            if i % 20 == 0:
                print(f"  Processing {i+1}/{len(queries)}...")

            # Embed query
            q_emb = self.embedder.embed(query)
            q_norm = q_emb / (np.linalg.norm(q_emb) + 1e-9)

            # Cosine similarity
            sims = np.dot(db_norm, q_norm)

            # Get top-k matches
            top_k_idx = np.argsort(sims)[-k:][::-1]

            if k == 1:
                # Simple: return entities from top match
                predictions.append(self.db_entities[top_k_idx[0]])
            else:
                # k-NN voting: count entity occurrences across top-k
                entity_votes = Counter()
                for idx in top_k_idx:
                    for e in self.db_entities[idx]:
                        entity_votes[e] += 1

                # Return entities that appear in majority of top-k (at least k/2 votes)
                threshold = k / 2
                voted_entities = [e for e, count in entity_votes.items() if count >= threshold]

                # If no majority, return most voted entity
                if not voted_entities:
                    voted_entities = [entity_votes.most_common(1)[0][0]]

                predictions.append(voted_entities)

        return predictions

In [ ]:
# Run Approach 1: Pre-trained Retrieval
print("="*50)
print("APPROACH 1: Pre-trained (Retrieval)")
print("="*50)

retrieval = RetrievalClassifier()
retrieval.build_database(train_queries, train_descriptions, train_entities)

preds_retrieval = retrieval.predict(test_queries)
y_pred_retrieval = mlb.transform(preds_retrieval)

metrics_retrieval = evaluate(y_test, y_pred_retrieval, ALL_ENTITIES,
                             "Approach 1: Pre-trained (Retrieval)")

## 5. Approach 2: Fine-tuned (with Description Augmentation)

Augment training data: for each query, add ONE new entry with combined descriptions.

In [ ]:
class EntityDataset(Dataset):
    """Dataset for fine-tuning."""
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True, padding='max_length',
                             max_length=self.max_length, return_tensors='pt')
        return {
            'input_ids': enc['input_ids'].flatten(),
            'attention_mask': enc['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.float)
        }


class FineTunedClassifier:
    """Fine-tuned classifier with description augmentation."""
    
    def __init__(self, num_labels: int):
        self.num_labels = num_labels
        self.model = None
        self.tokenizer = None
    
    def train(self, queries: List[str], descriptions: List[str], 
              y_train: np.ndarray, epochs: int = 10):
        """Train with augmented data."""
        # Augment: original queries + description entries
        X_aug = queries.copy()
        y_aug = list(y_train)
        
        for desc, label in zip(descriptions, y_train):
            if desc.strip():  # Only add non-empty descriptions
                X_aug.append(desc)
                y_aug.append(label)
        
        y_aug = np.array(y_aug)
        print(f"Training data: {len(queries)} queries + {len(X_aug)-len(queries)} description entries = {len(X_aug)} total")
        
        self.tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            BASE_MODEL, num_labels=self.num_labels,
            problem_type="multi_label_classification"
        )
        
        # Split for validation
        X_t, X_v, y_t, y_v = train_test_split(X_aug, y_aug, test_size=0.15, random_state=42)
        
        train_ds = EntityDataset(X_t, y_t, self.tokenizer)
        val_ds = EntityDataset(X_v, y_v, self.tokenizer)
        
        args = TrainingArguments(
            output_dir='./model_output',
            num_train_epochs=epochs,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=16,
            warmup_steps=100,
            weight_decay=0.01,
            logging_steps=50,
            eval_strategy='epoch',
            save_strategy='epoch',
            load_best_model_at_end=True,
            report_to='none',
            fp16=torch.cuda.is_available(),
        )
        
        trainer = Trainer(
            model=self.model, args=args,
            train_dataset=train_ds, eval_dataset=val_ds
        )
        trainer.train()
        
        self.model.eval()
        self.model.to(DEVICE)
        print("Training complete!")
    
    def predict(self, queries: List[str], threshold: float = 0.5) -> List[List[str]]:
        """Predict entities for queries."""
        predictions = []
        
        for i, query in enumerate(queries):
            if i % 20 == 0:
                print(f"  Processing {i+1}/{len(queries)}...")
            
            inputs = self.tokenizer(query, return_tensors='pt', truncation=True,
                                    padding=True, max_length=128)
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
            
            with torch.no_grad():
                logits = self.model(**inputs).logits
                probs = torch.sigmoid(logits).cpu().numpy()[0]
            
            preds = [ALL_ENTITIES[j] for j, p in enumerate(probs) if p > threshold]
            if not preds:
                preds = [ALL_ENTITIES[np.argmax(probs)]]
            predictions.append(preds)
        
        return predictions

In [ ]:
# Run Approach 2: Fine-tuned
print("="*50)
print("APPROACH 2: Fine-tuned (Description Augmentation)")
print("="*50)

finetuned = FineTunedClassifier(num_labels=len(ALL_ENTITIES))
finetuned.train(train_queries, train_descriptions, y_train, epochs=10)

print("\nPredicting on test set...")
preds_finetuned = finetuned.predict(test_queries)
y_pred_finetuned = mlb.transform(preds_finetuned)

metrics_finetuned = evaluate(y_test, y_pred_finetuned, ALL_ENTITIES,
                             "Approach 2: Fine-tuned (Description Augmentation)")

## 6. Comparison

In [ ]:
print("\n" + "="*60)
print(f"COMPARISON: {BASE_MODEL}")
print("="*60)

comparison = pd.DataFrame({
    'Metric': ['Exact Match', 'F1 Micro', 'F1 Macro', 'Precision', 'Recall', 'Hamming Loss'],
    'Pre-trained (Retrieval)': [
        metrics_retrieval['exact_match'], metrics_retrieval['f1_micro'],
        metrics_retrieval['f1_macro'], metrics_retrieval['precision'],
        metrics_retrieval['recall'], metrics_retrieval['hamming_loss']
    ],
    'Fine-tuned (Augmented)': [
        metrics_finetuned['exact_match'], metrics_finetuned['f1_micro'],
        metrics_finetuned['f1_macro'], metrics_finetuned['precision'],
        metrics_finetuned['recall'], metrics_finetuned['hamming_loss']
    ]
})

comparison['Δ'] = comparison['Fine-tuned (Augmented)'] - comparison['Pre-trained (Retrieval)']
print(comparison.to_string(index=False))

print(f"\nF1 improvement: {metrics_finetuned['f1_micro'] - metrics_retrieval['f1_micro']:+.4f}")

In [ ]:
# Save results
comparison.to_csv('comparison_results.csv', index=False)
print("Results saved to comparison_results.csv")